### Preprocess

In [1]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics.data.augment import LetterBox
import torch

In [2]:
def preprocess(im: torch.Tensor | list[np.ndarray]) -> torch.Tensor:
    not_tensor = not isinstance(im, torch.Tensor)
    if not_tensor:
        im = np.stack(pre_transform(im))
        if im.shape[-1] == 3:
            im = im[..., ::-1]  # BGR to RGB
        im = im.transpose((0, 3, 1, 2))  # BHWC to BCHW, (n, 3, h, w)
        im = np.ascontiguousarray(im)  # contiguous
        im = torch.from_numpy(im)

    im = im.to(device)
    im = im.half() if model_fp16 else im.float()  # uint8 to fp16/32
    if not_tensor:
        im /= 255  # 0 - 255 to 0.0 - 1.0
    return im

def pre_transform(im: list[np.ndarray]) -> list[np.ndarray]:
    letterbox = LetterBox(
        (640, 640),
        auto=len({x.shape for x in im}) == 1,
        stride=32,
    )
    return [letterbox(image=x) for x in im]

In [4]:
device = "cuda:0"
model_fp16 = False
image_path = "bus.jpg"

img = cv2.imread(image_path)
img = np.array([img])
im = preprocess(img)

### Load Model

In [5]:
import torch
from ultralytics.utils.torch_utils import attempt_compile, select_device
from ultralytics.nn.autobackend import AutoBackend

In [6]:
# Load Model
model = torch.load(
    "yolo26n.pth",
    map_location="cuda",
    weights_only=False,
)

model.eval()

# Setup Model
args_max_det = 300
args_agnostic_nms = False
args_device = None
verbose = False
args_dnn = False
args_data = "/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml"
args_half = False
args_compile = False

model_device = "cuda:0"
model_fp16 = False

if hasattr(model, "end2end"):
    if model.end2end:
        model.set_head_attr(max_det=args_max_det, agnostic_nms=args_agnostic_nms)
model = AutoBackend(
    model=model,
    device=select_device(args_device, verbose=verbose),
    dnn=args_dnn,
    data=args_data,
    fp16=args_half,
    fuse=True,
    verbose=verbose,
)
model.eval()
model = attempt_compile(model, device=model_device, mode=args_compile)

### Inference

In [7]:
preds = model(im, augment=False, visualize=False, embed=None)

In [11]:
len(preds)

2

In [12]:
preds[1]

{'one2many': {},
 'one2one': {'boxes': tensor([[[0.4898, 1.4011, 2.4434,  ..., 3.0844, 3.1138, 3.0075],
           [0.4779, 0.4868, 0.5140,  ..., 3.9861, 3.9789, 4.0386],
           [0.5213, 1.1931, 1.5597,  ..., 2.2834, 1.4222, 0.5120],
           [0.5544, 0.1925, 0.1479,  ..., 5.2861, 5.2463, 5.2574]]], device='cuda:0'),
  'scores': tensor([[[-14.7876, -17.0342, -17.0101,  ..., -16.6086, -19.3755, -19.7281],
           [-18.8656, -20.5721, -20.7013,  ..., -19.0372, -20.4660, -20.6337],
           [-17.2791, -18.9379, -19.0518,  ..., -19.7130, -21.6036, -21.5866],
           ...,
           [-17.6381, -20.3084, -20.2661,  ..., -17.6540, -19.3818, -19.0354],
           [-18.1885, -19.9796, -20.2478,  ..., -18.3715, -19.2919, -19.6249],
           [-17.4466, -20.9856, -21.3313,  ..., -19.8828, -21.2412, -20.8053]]], device='cuda:0'),
  'feats': [tensor([[[[ 0.1818,  0.2902,  0.2811,  ...,  0.3130,  0.2876,  0.3903],
             [ 0.3813,  0.3200,  0.2643,  ...,  0.2053,  0.4609,  0.494

In [13]:
type(preds[0])

torch.Tensor

In [14]:
type(preds[1])

dict

In [15]:
torch.save(preds, "tmp/preds.pt")

In [18]:
# torch.load("tmp/preds.pt", weights_only=False)